# 01 : Préparation des données - CMS Open Payments

**Dataset** : CMS Open Payments (General Payments)
**Source** : US Centers for Medicare & Medicaid Services, données publiques via API
**Périmètre** : Virginie-Occidentale (WV), années 2022 et 2023
**Volume** : ~186 000 paiements | ~5 700 professionnels de santé (HCP)

---

## Contexte : qu'est-ce que CMS Open Payments ?

Aux États-Unis, la loi (Sunshine Act) oblige les laboratoires pharmaceutiques et fabricants de dispositifs médicaux à **déclarer publiquement chaque paiement ou avantage** versé à un professionnel de santé : honoraires de conseil, interventions comme orateur, repas, voyages, formation. C'est **le** jeu de données de référence sur la relation commerciale industrie / professionnels de santé. Une ligne = un paiement.

## Objectif du notebook

Transformer ces paiements bruts en un jeu de données prêt à modéliser :
1. récupérer une tranche maîtrisée via l'**API CMS** (un État, deux années) ;
2. **auditer la qualité** (types, manquants, cohérence) ;
3. **agréger au niveau du professionnel de santé** (profil d'engagement) ;
4. construire la **cible de rétention** : le HCP payé en 2022 l'est-il encore en 2023 ?

Sortie : `data/openpayments.sqlite` (table `payments` = paiements bruts, table `hcp_features` = profils + cible).

## Variables clés utilisées

| Colonne | Signification |
| --- | --- |
| `covered_recipient_profile_id` | identifiant du professionnel de santé |
| `recipient_state` | État |
| `covered_recipient_specialty_1` | spécialité |
| `applicable_manufacturer_..._name` | laboratoire payeur |
| `total_amount_of_payment_usdollars` | montant du paiement (USD) |
| `nature_of_payment_or_transfer_of_value` | nature (repas, voyage, conseil, ...) |
| `program_year` | année (2022 ou 2023) |

## 0. Configuration

On fixe le périmètre (État, années), les identifiants des datasets CMS par année, et la liste des colonnes à conserver. `PAGE = 500` car l'API CMS **plafonne à 500 lignes par requête** : on paginera.

In [1]:
import sqlite3, pathlib
import pandas as pd
import requests

API = "https://openpaymentsdata.cms.gov/api/1"
STATE = "WV"
YEAR_FEATURES, YEAR_TARGET = 2022, 2023
DATASET_IDS = {2022: "df01c2f8-dc1f-4e79-96cb-8208beaf143c",
               2023: "fb3a65aa-c901-4a38-a813-b04b00dfa2a9",
               2024: "e6b17c6a-2534-4207-a4a1-6746a14911ff"}
KEEP = ["covered_recipient_profile_id", "covered_recipient_type", "recipient_state",
        "covered_recipient_specialty_1",
        "applicable_manufacturer_or_applicable_gpo_making_payment_name",
        "total_amount_of_payment_usdollars", "nature_of_payment_or_transfer_of_value", "program_year"]
PAGE = 500  # l'API CMS plafonne limit a 500

ROOT = pathlib.Path.cwd().parent if (pathlib.Path.cwd().parent / "data").exists() else pathlib.Path.cwd()
DB_PATH = ROOT / "data" / "openpayments.sqlite"
print("Projet:", ROOT)

Projet: C:\Juliette Vanessa\Desktop\notebook\pharma-commercial-genai


## 1. Récupération via l'API CMS

L'API renvoie les résultats par pages de 500 lignes. On filtre côté serveur sur l'État (`recipient_state`) pour ne télécharger que le périmètre utile (au lieu des ~15 millions de lignes annuelles), et on boucle sur `offset` jusqu'à tout récupérer.

In [2]:
def fetch_state_year(year, state):
    url = f"{API}/datastore/query/{DATASET_IDS[year]}/0"
    rows, offset, total = [], 0, None
    while True:
        params = {"limit": PAGE, "offset": offset,
                  "conditions[0][property]": "recipient_state",
                  "conditions[0][value]": state,
                  "conditions[0][operator]": "="}
        payload = requests.get(url, params=params, timeout=120).json()
        batch = payload.get("results", [])
        if total is None:
            total = payload.get("count", 0); print(f"  {year}/{state}: {total} lignes")
        if not batch:
            break
        rows.extend(batch); offset += len(batch)
        if offset >= total:
            break
    df = pd.DataFrame(rows)
    return df[[c for c in KEEP if c in df.columns]].copy()

## 2. Nettoyage et audit qualité

Avant toute analyse, on vérifie la donnée : conversion des montants et de l'année en numériques, extraction du **niveau haut** de la spécialité (le champ est hiérarchique, séparé par `|`), et exclusion des lignes sans identifiant (hôpitaux universitaires, hors périmètre HCP). L'audit imprime les effectifs, les manquants et les natures de paiement dominantes.

In [3]:
def clean(df):
    df = df.copy()
    df["total_amount_of_payment_usdollars"] = pd.to_numeric(df["total_amount_of_payment_usdollars"], errors="coerce")
    df["program_year"] = pd.to_numeric(df["program_year"], errors="coerce").astype("Int64")
    df["specialty"] = df["covered_recipient_specialty_1"].fillna("Unknown").str.split("|").str[0].str.strip()
    df = df[df["covered_recipient_profile_id"].notna()]
    df = df[df["covered_recipient_profile_id"].astype(str).str.len() > 0]
    return df

def quality_audit(df, label):
    print(f"=== Audit qualite - {label} ===")
    print("  lignes                :", len(df))
    print("  professionnels uniques:", df["covered_recipient_profile_id"].nunique())
    print("  montant manquant      :", int(df["total_amount_of_payment_usdollars"].isna().sum()))
    print("  montant total (USD)   :", round(df["total_amount_of_payment_usdollars"].sum()))

## 3. Agrégation : profil d'engagement + cible de rétention

On passe du **paiement** au **professionnel de santé** : pour chaque HCP, on résume son engagement en 2022 (nombre de paiements, montant total/moyen, nombre de labos, nombre de natures, part de chaque type de paiement, spécialité). La **cible** `retenu` vaut 1 si ce HCP reçoit au moins un paiement en 2023. Features en année N, cible en N+1 : pas de fuite temporelle.

In [4]:
def build_features(df_feat, ids_target):
    g = df_feat.groupby("covered_recipient_profile_id")
    feats = pd.DataFrame({
        "n_payments": g.size(),
        "total_amount": g["total_amount_of_payment_usdollars"].sum(),
        "mean_amount": g["total_amount_of_payment_usdollars"].mean(),
        "n_manufacturers": g["applicable_manufacturer_or_applicable_gpo_making_payment_name"].nunique(),
        "n_natures": g["nature_of_payment_or_transfer_of_value"].nunique(),
        "specialty": g["specialty"].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else "Unknown"),
        "state": g["recipient_state"].first(),
    })
    for nature, col in [("Food and Beverage", "share_food"), ("Travel and Lodging", "share_travel"),
                        ("Consulting Fee", "share_consulting"),
                        ("Compensation for services other than consulting, including serving as faculty or as a speaker at a venue other than a continuing education program", "share_speaker"),
                        ("Education", "share_education")]:
        part = df_feat[df_feat["nature_of_payment_or_transfer_of_value"] == nature].groupby("covered_recipient_profile_id").size()
        feats[col] = (part / feats["n_payments"]).reindex(feats.index).fillna(0.0)
    feats["retenu"] = feats.index.to_series().isin(ids_target).astype(int)
    return feats.reset_index()

## 4. Exécution du pipeline

On télécharge les deux années, on nettoie, on audite, on construit les profils + la cible, et on écrit la base SQLite.

In [5]:
df_n = clean(fetch_state_year(YEAR_FEATURES, STATE))
df_n1 = clean(fetch_state_year(YEAR_TARGET, STATE))
quality_audit(df_n, f"{STATE} {YEAR_FEATURES}")
quality_audit(df_n1, f"{STATE} {YEAR_TARGET}")

ids_target = set(df_n1["covered_recipient_profile_id"].unique())
features = build_features(df_n, ids_target)
print("Taux de retention:", round(features["retenu"].mean(), 3))

payments = pd.concat([df_n, df_n1], ignore_index=True)
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
with sqlite3.connect(DB_PATH) as con:
    payments.to_sql("payments", con, if_exists="replace", index=False)
    features.to_sql("hcp_features", con, if_exists="replace", index=False)
print("Base ecrite:", DB_PATH, "|", len(payments), "paiements,", len(features), "profils")

  2022/WV: 90294 lignes


  2023/WV: 96132 lignes


=== Audit qualite - WV 2022 ===
  lignes                : 90204
  professionnels uniques: 5707
  montant manquant      : 0
  montant total (USD)   : 7757148
=== Audit qualite - WV 2023 ===
  lignes                : 96019
  professionnels uniques: 6230
  montant manquant      : 0
  montant total (USD)   : 7553201


Taux de retention: 0.733


Base ecrite: C:\Juliette Vanessa\Desktop\notebook\pharma-commercial-genai\data\openpayments.sqlite | 186223 paiements, 5707 profils


> **Observations.**
> - WV compte ~90 000 paiements par an pour ~5 700 à 6 200 HCP, **aucun montant manquant**.
> - La nature **Food and Beverage** (repas) domine très largement : beaucoup de petits paiements fréquents.
> - **73,3 % des HCP de 2022 sont encore payés en 2023** : rétention élevée, donc tâche **déséquilibrée** (classe majoritaire = "retenu"), à garder en tête pour la modélisation.
> - Point de gouvernance repéré : les **noms de laboratoires ne sont pas normalisés** (« ABBVIE INC. » vs « AbbVie Inc. »), à corriger avant toute agrégation par labo.